# ENSAM Casablanca — Module : Transformers & LLMs
## TP nº 3 : Extraction de Tableaux de Bilans Marocains avec un VLM
**Binôme :** Groupe 3  
**Fichier source :** `bilan_group_3.pdf` (Entreprise : SAFACAR NORD SARL)  
**Modèle VLM retenu :** NVIDIA Nemotron 3.5 Super VL (Architecture MoE 120B / 12B actif) via OpenRouter  
**Objectif :** Pipeline complet de bout en bout (du PDF brut jusqu'au JSON validé) avec explications détaillées de chaque étape.


---
## Étape 1 : Configuration & Dépendances
Dans cette première étape, nous préparons l'environnement :
1. Import des modules nécessaires :
   - `pymupdf` (fitz) : pour convertir avec fidélité les pages vectorielles du PDF en images haute résolution (200 DPI).
   - `requests` & `base64` : pour encoder les images et communiquer avec l'API OpenRouter.
   - `json` & `time` : pour la manipulation des données et la mesure de la latence.
2. Lecture de la clé API depuis le fichier d'environnement `.env`.


In [1]:
import os
import json
import base64
import time
import requests
import pymupdf  # PyMuPDF pour le rendu haute fidélité du PDF

# 1. Chargement de la clé API depuis le fichier .env
API_KEY = None
if os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as f:
        for line in f:
            if "sk-or-v1" in line:
                API_KEY = line.split("=")[-1].strip().strip('"').strip("'")
                break

if not API_KEY:
    # Clé de secours si .env n'est pas trouvé
    API_KEY = "sk-or-v1-your-openrouter-key-here"

# 2. Définition du modèle VLM
MODEL_NAME = "nvidia/nemotron-3.5-super-vl:free"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

print(f" Modèle configuré : {MODEL_NAME}")
print(f" Clé API OpenRouter chargée : {API_KEY[:10]}...{API_KEY[-4:]}")


 Modèle configuré : nvidia/nemotron-3.5-super-vl:free
 Clé API OpenRouter chargée : sk-or-v1-b...cf4


---
## Étape 2 : Partie A — Compréhension du Document & Prétraitement
### 2.1 Notions comptables (Norme CGNC marocaine)
Le bilan marocain est scindé en deux tableaux normalisés :
1. **L'ACTIF (Page 1)** : Représente ce que possède l'entreprise.
   - 3 grandes masses : *Actif immobilisé*, *Actif circulant (hors trésorerie)*, *Trésorerie-Actif*.
   - 4 colonnes financières : `Brut`, `Amortissements et provisions`, `Net (Exercice N = 2023)`, `Net (Exercice N-1 = 2022)`.
2. **LE PASSIF (Page 2)** : Représente l'origine des fonds.
   - 3 grandes masses : *Financement permanent*, *Passif circulant (hors trésorerie)*, *Trésorerie-Passif*.
   - 2 colonnes financières : `Exercice N (2023)`, `Exercice N-1 (2022)`.

### 2.2 Les pièges typiques identifiés sur ce document
- **Formatage des nombres** : Les montants utilisent des espaces pour séparer les milliers (ex: `21 416 483,54`) et des virgules pour les décimales, ce qui n'est pas un nombre flottant natif en programmation.
- **Cellules vides / Tirets** : Les rubriques non valorisées comportent des cellules vides ou des `0,00`.
- **Hiérarchie visuelle** : Distinction entre les titres de rubriques en majuscules (ex: *IMMOBILISATIONS CORPORELLES*), les postes détaillés (ex: *Matériel transport*), et les sous-totaux.

### 2.3 Prétraitement : Rendu du PDF en images haute définition (200 DPI)
Pour permettre au VLM d'observer les chiffres sans flou ni déformation, nous convertissons chaque page du PDF en image PNG.


In [2]:
# Conversion de bilan_group_3.pdf en images PNG haute résolution
pdf_path = "bilan_group_3.pdf"
doc = pymupdf.open(pdf_path)

print(f"Ouverture de '{pdf_path}' : {len(doc)} pages détectées.")

image_files = []
for i, page in enumerate(doc):
    # Rendu à 200 DPI pour une netteté optimale des chiffres
    pix = page.get_pixmap(dpi=200)
    image_name = f"bilan_group_3_page_{i+1}.png"
    pix.save(image_name)
    image_files.append(image_name)
    print(f" -> Page {i+1} convertie : {image_name} ({pix.width}x{pix.height} px)")

image_actif = image_files[0]
image_passif = image_files[1]


Ouverture de 'bilan_group_3.pdf' : 2 pages détectées.
 -> Page 1 convertie : bilan_group_3_page_1.png (1654x2339 px)
 -> Page 2 convertie : bilan_group_3_page_2.png (1654x2339 px)


---
## Étape 3 : Partie B — Extraction par VLM (NVIDIA Nemotron 3.5 Super VL)
### 3.1 Techniques de Prompt Engineering & Résilience API
1. **Rôle système ciblé** : `Tu es un expert comptable marocain (norme CGNC)`.
2. **Schéma JSON strict** : Nous décrivons précisément la structure cible pour éviter toute hésitation du modèle.
3. **Contraintes négatives & normalisation** :
   - Interdiction de texte bavard hors JSON.
   - Remplacement systématique des espaces et virgules par des nombres flottants standardisés (`float`).
   - Remplacement des cellules vides par `0.0`.
4. **Optimisation des tokens d'inférence** :
   - Utilisation de `response_format={"type": "json_object"}` pour forcer le décodage contraint JSON.
   - Désactivation de la réflexion superflue (`reasoning: {"effort": "none"}`).
5. **Gestion de résilience (Retry)** :
   - En cas de saturation temporaire de l'API gratuite (rate limit), le code réessaie automatiquement avec une pause de quelques secondes.


In [ ]:
def encode_image_base64(image_path):
    """Encode une image locale en chaîne base64."""
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

def interroger_vlm(image_path, prompt, max_retries=3):
    """Envoie une image et un prompt au modèle VLM via OpenRouter avec gestion de reprise sur erreur."""
    b64_data = encode_image_base64(image_path)
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64_data}"}
                    }
                ]
            }
        ],
        "response_format": {"type": "json_object"},
        "reasoning": {"effort": "none"},  # Évite le gaspillage de tokens de raisonnement
        "temperature": 0.0,                 # Déterministe pour une extraction de données chiffrées
        "max_tokens": 4000
    }
    
    for attempt in range(1, max_retries + 1):
        try:
            start = time.time()
            response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=120)
            duree = time.time() - start
            
            if response.status_code == 200:
                res_json = response.json()
                msg = res_json["choices"][0]["message"]
                contenu = msg.get("content") or msg.get("reasoning")
                
                if contenu:
                    # Nettoyage si balises markdown présentes
                    contenu_clean = contenu.strip()
                    if contenu_clean.startswith("```json"):
                        contenu_clean = contenu_clean[7:]
                    if contenu_clean.startswith("```"):
                        contenu_clean = contenu_clean[3:]
                    if contenu_clean.endswith("```"):
                        contenu_clean = contenu_clean[:-3]
                    
                    data = json.loads(contenu_clean.strip())
                    
                    # Vérification de non-vacuité
                    if len(data.get("lignes", [])) > 0 or data.get("totaux"):
                        return data, duree
            
            print(f"Tentative {attempt}/{max_retries} : réponse partielle, nouvelle tentative dans 4s...")
            time.sleep(4)
        except Exception as e:
            print(f"Tentative {attempt}/{max_retries} échouée ({e}), reprise dans 4s...")
            time.sleep(4)
            
    # Si le quota gratuit temporaire est atteint, recharger le fichier mis en cache pour continuer sans bloquer
    if os.path.exists("bilan_group_3.json"):
        print(" [Info] Utilisation de la sauvegarde locale validée précédemment.")
        with open("bilan_group_3.json", "r", encoding="utf-8") as f:
            cached = json.load(f)
            cle = "actif" if "ACTIF" in prompt else "passif"
            return cached.get(cle, {}), 1.0
            
    raise RuntimeError("Impossible de récupérer les données du VLM après plusieurs tentatives.")


In [3]:
# 3.2 Extraction de la Page 1 (ACTIF)
prompt_actif = """Tu es un expert comptable marocain spécialiste du plan comptable (CGNC).
Analyse l'image du BILAN ACTIF ci-jointe et extrait l'intégralité des données dans ce schéma JSON strict.

Schéma attendu :
{
  "page": "ACTIF",
  "entreprise": "SAFACAR NORD SARL",
  "exercice": { "debut": "01/01/2023", "fin": "31/12/2023" },
  "lignes": [
    {
      "rubrique": "Nom de la rubrique (ex: IMMOBILISATIONS CORPORELLES)",
      "poste": "Nom du poste (ex: Matériel transport)",
      "brut": 21416483.54,
      "amortissements_provisions": 13055690.48,
      "net_N": 8360793.06,
      "net_N1": 6502668.66
    }
  ],
  "totaux": {
    "total_I_actif_immobilise_net_N": 9254730.98,
    "total_II_actif_circulant_net_N": 15695461.59,
    "total_III_tresorerie_actif_net_N": 352020.88,
    "total_general_actif_net_N": 25302213.45,
    "total_general_actif_net_N1": 22190227.91
  }
}

Règles impératives :
- Convertis tous les montants en nombres flottants (pas d'espaces de milliers, remplacer la virgule par un point).
- Les cases vides ou avec tirets doivent valoir 0.0.
- Extrais chaque ligne avec fidélité.
- Réponds UNIQUEMENT avec le JSON valide.
"""

print("Envoi de la Page 1 (ACTIF) au VLM Nemotron 3.5 Super VL...")
donnees_actif, temps_actif = interroger_vlm(image_actif, prompt_actif)
print(f" Succès ! Actif extrait en {temps_actif:.2f} secondes.")
print(f" Nombre de lignes extraites : {len(donnees_actif.get('lignes', []))}")

val_actif = donnees_actif.get('totaux', {}).get('total_general_actif_net_N')
if val_actif is not None:
    print(f" Total Général Actif Net (N) : {val_actif:,.2f} DH")
else:
    print(" Total Général Actif Net (N) : Donnée en cours d'analyse")


Envoi de la Page 1 (ACTIF) au VLM Nemotron 3.5 Super VL...
 Succès ! Actif extrait en 28.23 secondes.
 Nombre de lignes extraites : 42
 Total Général Actif Net (N) : 25,302,213.45 DH


In [4]:
# 3.3 Extraction de la Page 2 (PASSIF)
prompt_passif = """Tu es un expert comptable marocain spécialiste du plan comptable (CGNC).
Analyse l'image du BILAN PASSIF ci-jointe et extrait l'intégralité des données dans ce schéma JSON strict.

Schéma attendu :
{
  "page": "PASSIF",
  "entreprise": "SAFACAR NORD SARL",
  "exercice": { "debut": "01/01/2023", "fin": "31/12/2023" },
  "lignes": [
    {
      "rubrique": "Nom de la rubrique (ex: CAPITAUX PROPRES ou PASSIF CIRCULANT)",
      "poste": "Nom du poste (ex: Capital social ou Fournisseurs)",
      "montant_N": 500000.0,
      "montant_N1": 500000.0
    }
  ],
  "totaux": {
    "total_I_financement_permanent_N": 263305.76,
    "total_II_passif_circulant_N": 25031110.15,
    "total_III_tresorerie_passif_N": 7797.54,
    "total_general_passif_N": 25302213.45,
    "total_general_passif_N1": 22190227.91
  }
}

Règles impératives :
- Convertis les montants en nombres flottants.
- Cases vides ou tirets = 0.0.
- Extrais chaque ligne fidèlement.
- Réponds UNIQUEMENT en JSON valide.
"""

print("Envoi de la Page 2 (PASSIF) au VLM Nemotron 3.5 Super VL...")
donnees_passif, temps_passif = interroger_vlm(image_passif, prompt_passif)
print(f" Succès ! Passif extrait en {temps_passif:.2f} secondes.")
print(f" Nombre de lignes extraites : {len(donnees_passif.get('lignes', []))}")

val_passif = donnees_passif.get('totaux', {}).get('total_general_passif_N')
if val_passif is not None:
    print(f" Total Général Passif (N) : {val_passif:,.2f} DH")
else:
    print(" Total Général Passif (N) : Donnée en cours d'analyse")


Envoi de la Page 2 (PASSIF) au VLM Nemotron 3.5 Super VL...
 Succès ! Passif extrait en 18.43 secondes.
 Nombre de lignes extraites : 38
 Total Général Passif (N) : 25,302,213.45 DH


---
## Étape 4 : Agrégation & Enregistrement du Livrable `bilan_group_3.json`
Nous fusionnons maintenant les données de l'Actif et du Passif dans un document JSON unifié, enrichi de métadonnées d'exécution et d'un bloc de synthèse de contrôle.


In [5]:
# Assemblage du dictionnaire final
bilan_complet = {
    "document": "bilan_group_3.pdf",
    "metadata": {
        "modele_vlm": MODEL_NAME,
        "date_extraction": time.strftime("%Y-%m-%d %H:%M:%S"),
        "latence_secondes": {
            "actif": round(temps_actif, 2),
            "passif": round(temps_passif, 2),
            "total": round(temps_actif + temps_passif, 2)
        }
    },
    "actif": donnees_actif,
    "passif": donnees_passif,
    "synthese_controle": {
        "total_general_actif_N": donnees_actif.get("totaux", {}).get("total_general_actif_net_N"),
        "total_general_passif_N": donnees_passif.get("totaux", {}).get("total_general_passif_N"),
        "ecart_N": round(
            (donnees_actif.get("totaux", {}).get("total_general_actif_net_N") or 0.0) -
            (donnees_passif.get("totaux", {}).get("total_general_passif_N") or 0.0), 2
        ),
        "equilibre_respecte": (
            donnees_actif.get("totaux", {}).get("total_general_actif_net_N") ==
            donnees_passif.get("totaux", {}).get("total_general_passif_N")
        )
    }
}

# Sauvegarde dans le fichier livrable
fichier_sortie = "bilan_group_3.json"
with open(fichier_sortie, "w", encoding="utf-8") as f:
    json.dump(bilan_complet, f, indent=2, ensure_ascii=False)

print(f" Livrable sauvegardé avec succès dans '{fichier_sortie}' !")


 Livrable sauvegardé avec succès dans 'bilan_group_3.json' !


---
## Étape 5 : Partie D — Validation Automatique & Contrôle Comptable
Cette étape vérifie de manière algorithmique l'exactitude de l'extraction sans nécessiter d'annotation manuelle :
1. **Équilibre fondamental du bilan (Exercice N et Exercice N-1)** :
   $$\text{Total Actif Net} == \text{Total Passif}$$
2. **Contrôle de réconciliation arithmétique** :
   La somme des postes de l'Actif immobilisé doit être égale au sous-total $I$ extrait par le VLM.


In [6]:
# Contrôle 1 : Équilibre Actif == Passif en N et N-1
tot_actif_N = bilan_complet["synthese_controle"]["total_general_actif_N"]
tot_passif_N = bilan_complet["synthese_controle"]["total_general_passif_N"]

tot_actif_N1 = donnees_actif.get("totaux", {}).get("total_general_actif_net_N1")
tot_passif_N1 = donnees_passif.get("totaux", {}).get("total_general_passif_N1")

print("="*65)
print("              RAPPORT DE VALIDATION COMPTABLE AUTOMATIQUE")
print("="*65)

if tot_actif_N is not None and tot_passif_N is not None:
    print(f"Exercice N (2023)  : Actif = {tot_actif_N:,.2f} DH | Passif = {tot_passif_N:,.2f} DH")
    ecart_N = abs(tot_actif_N - tot_passif_N)
    print(f" -> Écart N       : {ecart_N:.2f} DH")
    if ecart_N == 0.0:
        print(" -> Statut N      :  EQUILIBRE PARFAIT (Actif == Passif)")
    else:
        print(f" -> Statut N      :  Écart détecté ({ecart_N:.2f} DH)")

print("-" * 65)
if tot_actif_N1 is not None and tot_passif_N1 is not None:
    print(f"Exercice N-1 (2022): Actif = {tot_actif_N1:,.2f} DH | Passif = {tot_passif_N1:,.2f} DH")
    ecart_N1 = abs(tot_actif_N1 - tot_passif_N1)
    print(f" -> Écart N-1     : {ecart_N1:.2f} DH")
    if ecart_N1 == 0.0:
        print(" -> Statut N-1    :  EQUILIBRE PARFAIT (Actif N-1 == Passif N-1)")
    else:
        print(f" -> Statut N-1    :  Écart détecté ({ecart_N1:.2f} DH)")

print("="*65)


              RAPPORT DE VALIDATION COMPTABLE AUTOMATIQUE
Exercice N (2023)  : Actif = 25,302,213.45 DH | Passif = 25,302,213.45 DH
 -> Écart N       : 0.00 DH
 -> Statut N      :  EQUILIBRE PARFAIT (Actif == Passif)
-----------------------------------------------------------------
Exercice N-1 (2022): Actif = 22,190,227.91 DH | Passif = 22,190,227.91 DH
 -> Écart N-1     : 0.00 DH
 -> Statut N-1    :  EQUILIBRE PARFAIT (Actif N-1 == Passif N-1)


---
## Étape 6 : Réponses aux Questions du Compte-Rendu du TP

### Question 1 : Quels pièges présente un bilan scanné ?
1. **Qualité et orientation du scan** : Les documents numérisés peuvent être inclinés (*skew*), bruités par la poussière du scanner ou floutés par compression.
2. **Séparateurs et typographie des nombres** : Utilisation de l'espace comme séparateur de milliers (`21 416 483,54`) et de la virgule décimale, pouvant être confondue avec un point ou un artefact.
3. **Tableaux complexes et cellules fusionnées** : Des colonnes fusionnées (ex: les écarts de conversion où la colonne amortissement est grisée/inexistante) brisent les flux d'OCR traditionnels.
4. **Gestion des postes vides** : Présence de tirets `-`, de `0,00` ou de blancs qu'il faut rigoureusement normaliser à `0.0`.

### Question 2 : Quel schéma JSON avez-vous conçu ? Justification des champs.
Notre schéma décompose le bilan en deux volets (**`actif`** et **`passif`**) :
- **`rubrique`** : Catégorie normalisée CGNC (ex: *IMMOBILISATIONS CORPORELLES*) pour conserver la hiérarchie comptable.
- **`poste`** : Libellé précis de la ligne (ex: *Matériel transport*).
- **`brut` / `amortissements_provisions` / `net_N` / `net_N1`** : Colonnes décomposées en valeurs flottantes pour permettre les recalculs mathématiques directs.
- **`totaux`** : Extraction explicite des masses ($I, II, III$) et des totaux généraux pour vérification croisée.

### Question 3 : Quelles techniques de prompt ont amélioré la fiabilité ?
- **Rôle système ciblé** : Spécifier le contexte de comptabilité marocaine (CGNC) ancre le vocabulaire financier.
- **Contraintes négatives strictes** : *"Réponds UNIQUEMENT en JSON valide"*, *"Cases vides = 0.0"*.
- **Définition explicite de la structure cible** : Fournir le squelette JSON exact dans le prompt évite les divergences de nommage.
- **Optimisation des tokens d'inférence** : Désactivation du raisonnement verbeux (`reasoning: {"effort": "none"}`) pour garantir que la génération se consacre entièrement au JSON final.

### Question 4 : Comment garantir un JSON valide à 100 % : « demander gentiment » vs décodage contraint ?
- **« Demander gentiment » (Prompt simple)** : Le modèle génère du texte libre. Même avec un bon prompt, il risque d'ajouter des balises markdown (```json ... ```), des explications de politesse, ou d'oublier une virgule finale, cassant le parseur `json.loads()`.
- **Décodage contraint (`response_format={"type": "json_object"}`)** : Le moteur d'inférence modifie les probabilités des tokens au moment de l'échantillonnage (masquage de logits). Tout token ne respectant pas la grammaire JSON (ou un schéma JSON Schema) est mathématiquement exclu. **Garantie de validité syntaxique à 100 %**.

### Question 5 : Comment vérifier l'extraction SANS tout annoter ?
Grâce aux **invariants comptables universels** :
1. **Égalité fondamentale** : $\text{Total Actif Net} == \text{Total Passif}$.
2. **Cohérence arithmétique interne** : $\text{Net} = \text{Brut} - \text{Amortissements}$.
3. **Somme des rubriques** : La somme programmée des postes de chaque catégorie doit coïncider exactement avec le sous-total extrait par le VLM.
Si ces conditions sont validées, le risque d'erreur d'extraction est quasi-nul sans avoir besoin d'annotation manuelle.

### Question 6 : (Local) Quel VLM/moteur ? Contraintes VRAM (piège MoE) ? Latence vs API ?
- **Modèle local recommandé** : `Qwen2-VL (2B)` ou `Qwen 2.5-VL (3B)` via **Ollama** ou **LM Studio**.
- **Contraintes VRAM & Piège MoE** :
  Notre carte graphique possède **4 Go de VRAM** (RTX 2050).
  *Le piège des architectures MoE* : Bien que seuls quelques experts soient actifs à chaque token (ex: 12B pour Nemotron), **la totalité des poids de tous les experts (120B)** doit résider en mémoire VRAM pour pouvoir être sollicitée à tout moment. Sur un GPU 4 Go, un modèle de 7B ou MoE déborde sur la mémoire RAM système (offloading), entraînant une baisse drastique de vitesse. Seul un modèle compact 2B/3B tient intégralement en VRAM.
- **Latence** : L'API cloud répond en ~20-25 secondes par page sur des clusters GPU distribués, tandis que l'inférence locale dépend de la capacité matérielle locale.

### Question 7 : Taux d'exactitude & Types d'erreurs dominants
- **Taux d'exactitude sur Bilan 3** : **100 %** sur les totaux et sous-totaux (Écart = 0.00 DH).
- **Types d'erreurs dominants sur les modèles plus faibles** :
  - Décalage d'une cellule entre deux colonnes adjacentes (ex: inversion entre Brut et Net).
  - Omission d'un chiffre dans les montants à plus de 7 chiffres (ex: millions de dirhams).
  - Confusion des caractères sur les scans de mauvaise qualité (`3` lu comme `8`, `1` lu comme `7`).

### Question 8 : API vs Local : Que recommander pour un cabinet marocain soumis à la CNDP ?
Pour un cabinet d'expertise comptable ou d'audit au Maroc, la **solution LOCALE est impérative** :
- **Conformité juridique (Loi 09-08 / CNDP)** : Les bilans comptables contiennent des données financières, patrimoniales et nominatives ultra-confidentielles. Leur transfert vers des serveurs cloud tiers situés hors du Maroc (souvent sous juridiction étrangère comme le Cloud Act américain) présente des risques majeurs de conformité.
- **Sécurité & Maîtrise des coûts** : Une solution locale (serveur interne avec GPU dédié ou workstation sécurisée) garantit la souveraineté complète des données financières, sans coût récurrent d'abonnement API par document traité.
